## Boilerplate

In [39]:
import numpy as np
from tqdm import tqdm
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision.transforms as transforms
import pickle

In [40]:
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1.25)

In [41]:
import medmnist
from medmnist import INFO, Evaluator

In [42]:
# Import local files
%load_ext autoreload
%autoreload 2
from bcnn_training import *
from cnn_training import get_semi_supervised_labels, SSLDataset
from plots import *
from variational_cnn import *
from constants import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [43]:
data_flag = 'dermamnist'
download = True

info = INFO[data_flag]
task = info['task']
n_channels = info['n_channels']
n_classes = len(info['label'])

DataClass = getattr(medmnist, info['python_class'])

In [44]:
# Note: this preprocesses data such that it has mean 0.5 and std dev 0.5.
data_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[.5], std=[.5])
])

# load the data
train_dataset = DataClass(split='train', transform=data_transform, download=download)
val_dataset = DataClass(split='val', transform=data_transform, download=download)
test_dataset = DataClass(split='test', transform=data_transform, download=download)

BATCH_SIZE = 128

# encapsulate data into dataloader form
train_loader = data.DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = data.DataLoader(dataset=val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = data.DataLoader(dataset=test_dataset, batch_size=2*BATCH_SIZE, shuffle=False)

In [9]:
def default_setup(lr=0.001, l2_weight=0.0, rho_init=-5.0):
    model = VariationalCNN(n_channels, n_classes, rho_init=rho_init)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=l2_weight)
    return model, criterion, optimizer

In [10]:
RANDOM_SEED = 42

## 50%

In [11]:
# Create SSL versions of our datasets
unlabeled_rate = 0.5

train_labels_ssl_50 = get_semi_supervised_labels(train_dataset, unlabeled_rate=unlabeled_rate, seed=RANDOM_SEED)
train_ssl_dataset_50 = SSLDataset(train_dataset, train_labels_ssl_50)
train_ssl_loader_50 = data.DataLoader(train_ssl_dataset_50, batch_size=BATCH_SIZE, shuffle=True)

Unlabeled rate: 0.5 | Total examples: 7007 | Labeled examples: 3506 | Unlabeled examples: 3501
Class 0: 114/228 labeled, 114 unlabeled
Class 1: 180/359 labeled, 179 unlabeled
Class 2: 385/769 labeled, 384 unlabeled
Class 3: 40/80 labeled, 40 unlabeled
Class 4: 390/779 labeled, 389 unlabeled
Class 5: 2347/4693 labeled, 2346 unlabeled
Class 6: 50/99 labeled, 49 unlabeled


In [12]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cpu


In [ ]:
all_histories_soft_50 = []
betas = [0.1, 1.0, 10.0, 100.0]
lrs = [0.0001, 0.001, 0.01]
for lr in lrs:
    for beta in betas:
        np.random.seed(RANDOM_SEED)
        torch.manual_seed(RANDOM_SEED)
        model, criterion, optimizer = default_setup(lr=lr, l2_weight=0.0, rho_init=-5.0)
        model = model.to(device)
        history = train_loop_bcnn_soft_pseudo_label(
            model, train_ssl_loader_50, val_loader, criterion, optimizer,
            num_epochs=20, alpha=0.5, beta=beta, num_samples=10, device=device)
        best_history = max(history, key=lambda x: x['val_auc_macro'])
        all_histories_soft_50.append({
            "lr": lr,
            "beta": beta,
            "history": history,
            "best_history": best_history,
        })
        print(f"Finished {len(all_histories_soft_50)}/{len(betas) * len(lrs)}: lr={lr} | beta={beta} | Val AUC Macro: {best_history['val_auc_macro']:.4f} | Val NLL Macro: {best_history['val_macro_nll']:.4f}")

In [21]:
# Ran above on colab GPU and saved results, so we can load them here
with open('results/upgrade/all_histories_soft_50.pkl', 'rb') as f:
    all_histories_soft_50 = pickle.load(f)

In [22]:
for h in sorted(all_histories_soft_50, key=lambda x: x['best_history']['val_auc_macro'], reverse=True):
    print(f"LR: {h['lr']} | Beta: {h['beta']} | Best Val AUC Macro: {h['best_history']['val_auc_macro']:.4f} | Val NLL Macro: {h['best_history']['val_macro_nll']:.4f} ")

LR: 0.001 | Beta: 100.0 | Best Val AUC Macro: 0.9113 | Val NLL Macro: 1.9362 
LR: 0.001 | Beta: 0.1 | Best Val AUC Macro: 0.9104 | Val NLL Macro: 1.5463 
LR: 0.001 | Beta: 1.0 | Best Val AUC Macro: 0.9097 | Val NLL Macro: 1.6101 
LR: 0.001 | Beta: 10.0 | Best Val AUC Macro: 0.9080 | Val NLL Macro: 1.5543 
LR: 0.0001 | Beta: 100.0 | Best Val AUC Macro: 0.8973 | Val NLL Macro: 1.9432 
LR: 0.0001 | Beta: 0.1 | Best Val AUC Macro: 0.8971 | Val NLL Macro: 1.9371 
LR: 0.0001 | Beta: 10.0 | Best Val AUC Macro: 0.8967 | Val NLL Macro: 1.9721 
LR: 0.0001 | Beta: 1.0 | Best Val AUC Macro: 0.8963 | Val NLL Macro: 1.9703 
LR: 0.01 | Beta: 1.0 | Best Val AUC Macro: 0.8854 | Val NLL Macro: 1.9156 
LR: 0.01 | Beta: 0.1 | Best Val AUC Macro: 0.8827 | Val NLL Macro: 2.1183 
LR: 0.01 | Beta: 10.0 | Best Val AUC Macro: 0.8763 | Val NLL Macro: 2.0916 
LR: 0.01 | Beta: 100.0 | Best Val AUC Macro: 0.8567 | Val NLL Macro: 2.2615 


so lr=0.001, beta=100 had best mAUC, but lr=0.001 beta=0.1 had similar results with better mNLL

TODO: do we always just blindly pick best mAUC, like in this situation here? seems arbitrary if not, but like the second one kinda feels like the better option here

In [ ]:
# get test metrics on 3 seeds
SEEDS = [42, 123, 145]
test_results_by_seed_soft_50 = {}
for seed in SEEDS:
    np.random.seed(seed)
    torch.manual_seed(seed)
    lr, beta = 0.001, 100.0
    model, criterion, optimizer = default_setup(lr=lr, l2_weight=0.0, rho_init=-5.0)
    model = model.to(device)
    history = train_loop_bcnn_soft_pseudo_label(
            model, train_ssl_loader_50, val_loader, criterion, optimizer,
            num_epochs=20, alpha=0.5, beta=beta, num_samples=10, device=device)
    best_history = max(history, key=lambda x: x['val_auc_macro'])
    best_model = VariationalCNN(n_channels, n_classes)
    best_model = best_model.to(device)
    best_model.load_state_dict(best_history['model_state'])
    test_results = evaluate_bayesian(best_model, test_loader, device=device, mc_samples=20)
    test_results_by_seed_soft_50[seed] = test_results

In [45]:
# Ran above in colab GPU and saved results to file
with open('results/upgrade/test_results_by_seed_soft_50.pkl', 'rb') as f:
    test_results_by_seed_soft_50 = pickle.load(f)

In [46]:
print_aggregate_test_results(test_results_by_seed_soft_50)

Test mAUC: 0.8962 ± 0.0067
Test mNLL: 1.7251 ± 0.1295
Test per-class NLL: [1.9190 ± 0.3600  1.4546 ± 0.0176  1.5760 ± 0.0889  3.5118 ± 0.2726  1.4434 ± 0.2695  0.3426 ± 0.1100  1.8284 ± 0.2552]
Test per-class AUC: [0.9169 ± 0.0142 |  0.9167 ± 0.0036 |  0.8611 ± 0.0068 |  0.8793 ± 0.0171 |  0.8412 ± 0.0042 |  0.8941 ± 0.0067 |  0.9640 ± 0.0086 |]


## 75%

In [18]:
# Create SSL versions of our datasets
unlabeled_rate = 0.75

train_labels_ssl_75 = get_semi_supervised_labels(train_dataset, unlabeled_rate=unlabeled_rate, seed=RANDOM_SEED)
train_ssl_dataset_75 = SSLDataset(train_dataset, train_labels_ssl_75)
train_ssl_loader_75 = data.DataLoader(train_ssl_dataset_75, batch_size=BATCH_SIZE, shuffle=True)

Unlabeled rate: 0.75 | Total examples: 7007 | Labeled examples: 1754 | Unlabeled examples: 5253
Class 0: 57/228 labeled, 171 unlabeled
Class 1: 90/359 labeled, 269 unlabeled
Class 2: 193/769 labeled, 576 unlabeled
Class 3: 20/80 labeled, 60 unlabeled
Class 4: 195/779 labeled, 584 unlabeled
Class 5: 1174/4693 labeled, 3519 unlabeled
Class 6: 25/99 labeled, 74 unlabeled


In [ ]:
all_histories_soft_75 = []
betas = [0.1, 1.0, 10.0, 100.0]
lrs = [0.0001, 0.001, 0.01]
for lr in lrs:
    for beta in betas:
        np.random.seed(RANDOM_SEED)
        torch.manual_seed(RANDOM_SEED)
        model, criterion, optimizer = default_setup(lr=lr, l2_weight=0.0, rho_init=-5.0)
        model = model.to(device)
        history = train_loop_bcnn_soft_pseudo_label(
            model, train_ssl_loader_75, val_loader, criterion, optimizer,
            num_epochs=20, alpha=0.5, beta=beta, num_samples=10, device=device)
        best_history = max(history, key=lambda x: x['val_auc_macro'])
        all_histories_soft_75.append({
            "lr": lr,
            "beta": beta,
            "history": history,
            "best_history": best_history,
        })
        print(f"Finished {len(all_histories_soft_75)}/{len(betas) * len(lrs)}: lr={lr} | beta={beta} | Val AUC Macro: {best_history['val_auc_macro']:.4f} | Val NLL Macro: {best_history['val_macro_nll']:.4f}")

In [20]:
# Ran above on colab GPU and saved results, so we can load them here
with open('results/upgrade/all_histories_soft_75.pkl', 'rb') as f:
    all_histories_soft_75 = pickle.load(f)

In [23]:
for h in sorted(all_histories_soft_75, key=lambda x: x['best_history']['val_auc_macro'], reverse=True):
    print(f"LR: {h['lr']} | Beta: {h['beta']} | Best Val AUC Macro: {h['best_history']['val_auc_macro']:.4f} | Val NLL Macro: {h['best_history']['val_macro_nll']:.4f} ")

LR: 0.001 | Beta: 100.0 | Best Val AUC Macro: 0.8999 | Val NLL Macro: 1.9470 
LR: 0.001 | Beta: 0.1 | Best Val AUC Macro: 0.8992 | Val NLL Macro: 2.0108 
LR: 0.001 | Beta: 1.0 | Best Val AUC Macro: 0.8958 | Val NLL Macro: 2.0428 
LR: 0.001 | Beta: 10.0 | Best Val AUC Macro: 0.8940 | Val NLL Macro: 1.9567 
LR: 0.0001 | Beta: 10.0 | Best Val AUC Macro: 0.8745 | Val NLL Macro: 2.0394 
LR: 0.0001 | Beta: 100.0 | Best Val AUC Macro: 0.8737 | Val NLL Macro: 2.0189 
LR: 0.0001 | Beta: 1.0 | Best Val AUC Macro: 0.8734 | Val NLL Macro: 1.9192 
LR: 0.0001 | Beta: 0.1 | Best Val AUC Macro: 0.8709 | Val NLL Macro: 1.9440 
LR: 0.01 | Beta: 0.1 | Best Val AUC Macro: 0.8674 | Val NLL Macro: 2.1252 
LR: 0.01 | Beta: 1.0 | Best Val AUC Macro: 0.8662 | Val NLL Macro: 2.1807 
LR: 0.01 | Beta: 10.0 | Best Val AUC Macro: 0.8622 | Val NLL Macro: 2.1600 
LR: 0.01 | Beta: 100.0 | Best Val AUC Macro: 0.8493 | Val NLL Macro: 2.3533 


In [ ]:
SEEDS = [42, 123, 145]
test_results_by_seed_soft_75 = {}
for seed in SEEDS:
    np.random.seed(seed)
    torch.manual_seed(seed)
    lr, beta = 0.001, 100.0
    model, criterion, optimizer = default_setup(lr=lr, l2_weight=0.0, rho_init=-5.0)
    model = model.to(device)
    history = train_loop_bcnn_soft_pseudo_label(
            model, train_ssl_loader_75, val_loader, criterion, optimizer,
            num_epochs=20, alpha=0.5, beta=beta, num_samples=10, device=device)
    best_history = max(history, key=lambda x: x['val_auc_macro'])
    best_model = VariationalCNN(n_channels, n_classes)
    best_model = best_model.to(device)
    best_model.load_state_dict(best_history['model_state'])
    test_results = evaluate_bayesian(best_model, test_loader, device=device, mc_samples=20)
    test_results_by_seed_soft_75[seed] = test_results

In [47]:
# Ran above in colab GPU and saved results to file
with open('results/upgrade/test_results_by_seed_soft_75.pkl', 'rb') as f:
    test_results_by_seed_soft_75 = pickle.load(f)

In [48]:
print_aggregate_test_results(test_results_by_seed_soft_75)

Test mAUC: 0.8811 ± 0.0029
Test mNLL: 1.9825 ± 0.1176
Test per-class NLL: [2.2008 ± 0.1837  2.0337 ± 0.1939  1.1016 ± 0.0602  3.4693 ± 0.5889  1.7528 ± 0.3573  0.3533 ± 0.0856  2.9662 ± 0.3465]
Test per-class AUC: [0.9235 ± 0.0017 |  0.9078 ± 0.0043 |  0.8500 ± 0.0099 |  0.8608 ± 0.0123 |  0.8227 ± 0.0183 |  0.8857 ± 0.0024 |  0.9175 ± 0.0111 |]


## 90%

In [24]:
# Create SSL versions of our datasets
unlabeled_rate = 0.9

train_labels_ssl_90 = get_semi_supervised_labels(train_dataset, unlabeled_rate=unlabeled_rate, seed=RANDOM_SEED)
train_ssl_dataset_90 = SSLDataset(train_dataset, train_labels_ssl_90)
train_ssl_loader_90 = data.DataLoader(train_ssl_dataset_90, batch_size=BATCH_SIZE, shuffle=True)

Unlabeled rate: 0.9 | Total examples: 7007 | Labeled examples: 702 | Unlabeled examples: 6305
Class 0: 23/228 labeled, 205 unlabeled
Class 1: 36/359 labeled, 323 unlabeled
Class 2: 77/769 labeled, 692 unlabeled
Class 3: 8/80 labeled, 72 unlabeled
Class 4: 78/779 labeled, 701 unlabeled
Class 5: 470/4693 labeled, 4223 unlabeled
Class 6: 10/99 labeled, 89 unlabeled


In [ ]:
all_histories_soft_90 = []
betas = [0.1, 1.0, 10.0, 100.0]
lrs = [0.0001, 0.001, 0.01]
for lr in lrs:
    for beta in betas:
        np.random.seed(RANDOM_SEED)
        torch.manual_seed(RANDOM_SEED)
        model, criterion, optimizer = default_setup(lr=lr, l2_weight=0.0, rho_init=-5.0)
        model = model.to(device)
        history = train_loop_bcnn_soft_pseudo_label(
            model, train_ssl_loader_90, val_loader, criterion, optimizer,
            num_epochs=20, alpha=0.5, beta=beta, num_samples=10, device=device)
        best_history = max(history, key=lambda x: x['val_auc_macro'])
        all_histories_soft_90.append({
            "lr": lr,
            "beta": beta,
            "history": history,
            "best_history": best_history,
        })
        print(f"Finished {len(all_histories_soft_90)}/{len(betas) * len(lrs)}: lr={lr} | beta={beta} | Val AUC Macro: {best_history['val_auc_macro']:.4f} | Val NLL Macro: {best_history['val_macro_nll']:.4f}")

In [32]:
# Ran above on colab GPU and saved results, so we can load them here
with open('results/upgrade/all_histories_soft_90.pkl', 'rb') as f:
    all_histories_soft_90 = pickle.load(f)

In [26]:
for h in sorted(all_histories_soft_90, key=lambda x: x['best_history']['val_auc_macro'], reverse=True):
    print(f"LR: {h['lr']} | Beta: {h['beta']} | Best Val AUC Macro: {h['best_history']['val_auc_macro']:.4f} | Val NLL Macro: {h['best_history']['val_macro_nll']:.4f} ")

LR: 0.001 | Beta: 100.0 | Best Val AUC Macro: 0.8705 | Val NLL Macro: 1.9447 
LR: 0.001 | Beta: 1.0 | Best Val AUC Macro: 0.8661 | Val NLL Macro: 2.1615 
LR: 0.001 | Beta: 10.0 | Best Val AUC Macro: 0.8644 | Val NLL Macro: 2.5015 
LR: 0.001 | Beta: 0.1 | Best Val AUC Macro: 0.8608 | Val NLL Macro: 2.0769 
LR: 0.0001 | Beta: 10.0 | Best Val AUC Macro: 0.8382 | Val NLL Macro: 2.5147 
LR: 0.0001 | Beta: 100.0 | Best Val AUC Macro: 0.8349 | Val NLL Macro: 2.3283 
LR: 0.0001 | Beta: 0.1 | Best Val AUC Macro: 0.8345 | Val NLL Macro: 2.3382 
LR: 0.0001 | Beta: 1.0 | Best Val AUC Macro: 0.8343 | Val NLL Macro: 2.3652 
LR: 0.01 | Beta: 10.0 | Best Val AUC Macro: 0.8341 | Val NLL Macro: 2.3890 
LR: 0.01 | Beta: 0.1 | Best Val AUC Macro: 0.8337 | Val NLL Macro: 2.3440 
LR: 0.01 | Beta: 1.0 | Best Val AUC Macro: 0.8309 | Val NLL Macro: 2.1496 
LR: 0.01 | Beta: 100.0 | Best Val AUC Macro: 0.7908 | Val NLL Macro: 2.9464 


In [ ]:
# TODO: probbaly not long enough for 0.0001 to converge (20 epochs)

In [ ]:
SEEDS = [42, 123, 145]
test_results_by_seed_soft_90 = {}
for seed in SEEDS:
    np.random.seed(seed)
    torch.manual_seed(seed)
    lr, beta = 0.001, 100.0
    model, criterion, optimizer = default_setup(lr=lr, l2_weight=0.0, rho_init=-5.0)
    model = model.to(device)
    history = train_loop_bcnn_soft_pseudo_label(
            model, train_ssl_loader_90, val_loader, criterion, optimizer,
            num_epochs=20, alpha=0.5, beta=beta, num_samples=10, device=device)
    best_history = max(history, key=lambda x: x['val_auc_macro'])
    best_model = VariationalCNN(n_channels, n_classes)
    best_model = best_model.to(device)
    best_model.load_state_dict(best_history['model_state'])
    test_results = evaluate_bayesian(best_model, test_loader, device=device, mc_samples=20)
    test_results_by_seed_soft_90[seed] = test_results

In [49]:
# Ran above in colab GPU and saved results to file
with open('results/upgrade/test_results_by_seed_soft_90.pkl', 'rb') as f:
    test_results_by_seed_soft_90 = pickle.load(f)

In [50]:
print_aggregate_test_results(test_results_by_seed_soft_90)

Test mAUC: 0.8489 ± 0.0017
Test mNLL: 2.1429 ± 0.0416
Test per-class NLL: [2.4537 ± 0.2104  2.2277 ± 0.3386  1.7524 ± 0.3369  3.6573 ± 0.3303  1.6878 ± 0.0085  0.2990 ± 0.0368  2.9221 ± 0.6601]
Test per-class AUC: [0.9039 ± 0.0129 |  0.8764 ± 0.0159 |  0.8356 ± 0.0091 |  0.8338 ± 0.0186 |  0.7787 ± 0.0028 |  0.8601 ± 0.0042 |  0.8536 ± 0.0259 |]
